# Training Data Cleaning and Split

In [15]:
import os
import sys
import logging
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from typing import Literal
from instanovo.transformer.dataset import remove_modifications as clean_peptide
# Fix this later, imports should work without this
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))


from common.utils import collect_files, get_or_create_folder
from common.logger import get_logger_config
from common.constants import (
    BASE_RAW_DATA_DIR,
    BASE_LOGS_DIR,
    BASE_PLOTS_DIR,
    BASE_REPORTS_CSV_DIR,
)

2025-04-09 11:57:53,935 - rdkit - INFO - Enabling RDKit 2024.09.6 jupyter extensions


In [4]:
logger_config = get_logger_config(subdir=None)
logging.config.dictConfig(logger_config)
logger = logging.getLogger(__name__)

## Remove entries with `precursor_charge` less than 2

## Check how do the peptides in the dataset here overlap the peptides from Kevin

In [6]:
peptides_file_paths = collect_files(BASE_REPORTS_CSV_DIR, ext="csv")

In [8]:
# TODO: Improve the `collect_files` function
peptides_file_paths = [
    path for path in peptides_file_paths if "unique_peptides" in path
]

assert peptides_file_paths, peptides_file_paths
# Grab all csv of interest but ATTENTION;
# loading all many csv files will increase the computation time
df = pd.concat([pd.read_csv(file) for file in peptides_file_paths], ignore_index=True)
df.head(20)

,Unique Peptides
0,HNGTGGR
1,SQNCHNSSSR
2,AAGMNHTK
3,ANASHDQPQK
4,HNDSGASECR
5,GGGGGGGGGGGGGSGSSSGSSTSR
6,RQQQQQQQQQQQQK
7,QQQQQQQQQQQQK
8,KNDSGAYR
9,KCLNHTTQK


In [9]:
df["Unique Peptides"].describe()

count                163595
unique                44976
top       AVCMLSNTTAIAEAWAR
freq                     10
Name: Unique Peptides, dtype: object

In [10]:
unique_peptides_df = df["Unique Peptides"].unique()
pd.DataFrame({"Unique Peptides": unique_peptides_df}).to_csv(
    BASE_REPORTS_CSV_DIR
    / f"overall_projects_unique_peptides_counting_{df['Unique Peptides'].nunique()}.csv",
    index=False,
)

### Overlap with peptides from identity files from Kevin

In [22]:
def compute_and_save_overlap(identify_file_path, other_peptides, split:Literal["train", "test", "valid"]|None = None, blacklist_file_path=None): # noqa

    """
    Computes and saves the overlap between peptides in a given identity file and a list of reference peptides.

    Args:
        identify_file_path (str or Path): CSV file path containing identified peptides with an optional 'split' column.
        other_peptides (set or list): Reference peptides to compare against (e.g., from the glyco project).
        split (Literal["train", "test", "valid"], optional): If set, filters peptides in the identity file by this split.
        blacklist_file_path (str or Path, optional): CSV file path containing peptides to exclude from the identity file.

    Saves:
        A CSV file listing the overlapping peptides, with filename reflecting the applied filters (split, blacklist).
    """
    identify_file_path = BASE_REPORTS_CSV_DIR / identify_file_path
    identity_df = pd.read_csv(identify_file_path)
    logger.info(f"Loaded {len(identity_df)} peptides from {identify_file_path}")
    
    if blacklist_file_path is not None:
        blacklist_df = pd.read_csv(blacklist_file_path)
        blacklist_peptides_set = set(blacklist_df["sequence"])
        identity_df = identity_df[~identity_df["sequence"].isin(blacklist_peptides_set)]
        logger.info(f"Filtered out {len(blacklist_peptides_set)} blacklisted peptides and leaving {len(identity_df)} peptides")    
    if split is not None:
        identity_df = identity_df[identity_df["split"] == split]
        logger.info(f"Got {len(identity_df)} peptides from {identify_file_path} after filtering by {split}")
    # Extract unique peptides from the 'sequence' column
    identity_unique_peptides = set(identity_df["sequence"].unique())
    if isinstance(other_peptides, str):
        other_path = BASE_REPORTS_CSV_DIR / other_peptides
        project_name = other_path.stem
        other_peptides = pd.read_csv(other_path)["sequence"].unique()
        logger.info(
            f"Loaded {len(other_peptides)} glyco projects peptides from {other_path}"
        )
    else:
        project_name = "glyco_projects"
    # Compute overlap now using the cleaned version of the peptides
    overlap_peptides =  identity_unique_peptides & {clean_peptide(p) for p in other_peptides}
    logger.info(f"Found {len(overlap_peptides)} {split or 'train/valid/test'} peptides overlap")
    
    # Save results
    column_name = f"Overlapped {split} peptides" if split else "Overlapped peptides"
    overlap_df = pd.DataFrame({column_name: list(overlap_peptides)})
    output_file_prefix = ""
    if split:
        output_file_prefix += f"{split}_"
    if blacklist_file_path:
        output_file_prefix += "blacklist_"
    output_file = BASE_REPORTS_CSV_DIR / (
        f"{output_file_prefix}overlap_{identify_file_path.stem}_{len(identity_unique_peptides)}_with_{project_name}_{len(other_peptides)}_found_{len(overlap_peptides)}.csv"
    )
    overlap_df.to_csv(output_file, index=False)

    logger.info(f"Saved: {output_file}")

In [23]:
# Compute and save overlaps without any specific filtering
if True:
    logger.info("Compute and save overlaps without any specific filtering")
    identity_files_from_kevin = [
        "identity_splits_proteome_tools_from_kevin.csv",
        "identity_splits_massivekb_from_kevin.csv",
        "identity_splits_blacklist_from_kevin.csv",
        "identity_splits_phospho_from_kevin.csv",
    ]
    
    for file_path in identity_files_from_kevin:
        compute_and_save_overlap(file_path, unique_peptides_df)

2025-04-09 12:33:47,911 - __main__ - INFO - Compute and save overlaps without any specific filtering
2025-04-09 12:33:48,652 - __main__ - INFO - Loaded 753915 peptides from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/identity_splits_proteome_tools_from_kevin.csv
2025-04-09 12:33:49,809 - __main__ - INFO - Found 10672 train/valid/test peptides overlap
2025-04-09 12:33:49,827 - __main__ - INFO - Saved: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/overlap_identity_splits_proteome_tools_from_kevin_753915_with_glyco_projects_44976_found_10672.csv
2025-04-09 12:33:51,488 - __main__ - INFO - Loaded 1608967 peptides from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/identity_splits_massivekb_from_kevin.csv
2025-04-09 12:33:52,520 - __main__ - INFO - Found 20130 train/valid/test peptides overlap
2025-04-09 12:33:52,551 - __main__ - INFO - Saved: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoG

In [25]:
# Compute and save overlaps for train/valid/test but ignoring blacklisted peptides
if True:
    logger.info("Compute and save overlaps for train/valid/test but ignoring blacklisted peptides")
    identity_files_from_kevin = [
        "identity_splits_proteome_tools_from_kevin.csv",
        "identity_splits_massivekb_from_kevin.csv",
        "identity_splits_phospho_from_kevin.csv",
        # "identity_splits_blacklist_from_kevin.csv",
    ]
    
    for file_path in identity_files_from_kevin:
        for split in ("train", "valid", "test"):
            compute_and_save_overlap(file_path, unique_peptides_df, split=split)

2025-04-09 12:34:44,992 - __main__ - INFO - Compute and save overlaps for train/valid/test but ignoring blacklisted peptides
2025-04-09 12:34:46,316 - __main__ - INFO - Loaded 753915 peptides from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/identity_splits_proteome_tools_from_kevin.csv
2025-04-09 12:34:46,474 - __main__ - INFO - Got 678525 peptides from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/identity_splits_proteome_tools_from_kevin.csv after filtering by train
2025-04-09 12:34:47,411 - __main__ - INFO - Found 7662 train peptides overlap
2025-04-09 12:34:47,431 - __main__ - INFO - Saved: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/train_overlap_identity_splits_proteome_tools_from_kevin_678525_with_glyco_projects_44976_found_7662.csv
2025-04-09 12:34:48,429 - __main__ - INFO - Loaded 753915 peptides from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/iden

In [26]:
# Compute and save overlaps for train/valid/test and accounting for blacklisted peptides
if True:
    logger.info("Compute and save overlaps for train/valid/test and accounting for blacklisted peptides")
    identity_files_from_kevin = [
        "identity_splits_proteome_tools_from_kevin.csv",
        "identity_splits_massivekb_from_kevin.csv",
        "identity_splits_phospho_from_kevin.csv",
        # Does not make sens to iterate over blacklisted peptides here
        # "identity_splits_blacklist_from_kevin.csv",
    ]

    for file_path in identity_files_from_kevin:
        for split in ("train", "valid", "test"):
            compute_and_save_overlap(file_path, unique_peptides_df, split=split, blacklist_file_path=BASE_REPORTS_CSV_DIR / "identity_splits_blacklist_from_kevin.csv")


2025-04-09 12:36:21,611 - __main__ - INFO - Compute and save overlaps for train/valid/test and accounting for blacklisted peptides
2025-04-09 12:36:22,804 - __main__ - INFO - Loaded 753915 peptides from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/identity_splits_proteome_tools_from_kevin.csv
2025-04-09 12:36:23,935 - __main__ - INFO - Filtered out 248900 blacklisted peptides and leaving 678525 peptides
2025-04-09 12:36:24,080 - __main__ - INFO - Got 678525 peptides from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/identity_splits_proteome_tools_from_kevin.csv after filtering by train
2025-04-09 12:36:25,115 - __main__ - INFO - Found 7662 train peptides overlap
2025-04-09 12:36:25,136 - __main__ - INFO - Saved: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/train_blacklist_overlap_identity_splits_proteome_tools_from_kevin_678525_with_glyco_projects_44976_found_7662.csv
2025-04-09 12:36:26,019 

## Split without Kevin constaint

In [12]:
unique_peptides_df

44976

In [13]:
from instanovo.utils.data_handler import SpectrumDataFrame

[04/07/25 11:58:30] INFO     Enabling RDKit 2024.09.6 jupyter extensions                             ]8;id=481547;file:///home/hjisaac/.cache/pypoetry/virtualenvs/instanovoglyco-u5tn6RZG-py3.10/lib/python3.10/site-packages/rdkit/__init__.py\__init__.py]8;;\:]8;id=419519;file:///home/hjisaac/.cache/pypoetry/virtualenvs/instanovoglyco-u5tn6RZG-py3.10/lib/python3.10/site-packages/rdkit/__init__.py#22\22]8;;\

In [14]:
SpectrumDataFrame

instanovo.utils.data_handler.SpectrumDataFrame